## Local Volatility Calibration Using Dupire’s Formula

The Local Volatility Model is a non-parametric model that allows volatility to be a function of both time and moneyness  $\sigma_{loc}(S, t)$. It provides a way to construct a volatility surface directly from market option prices without assuming a specific stochastic process.

Key Advantages:

✅ Perfectly fits market-implied volatility surfaces (no model risk).

✅ Captures term structure and moneyness dependence of volatility.

✅ Better pricing of exotic options (barriers, cliquets, forward-start options).

### Theory: Dupire’s Local Volatility Formula

Given a set of market-implied volatilities  \sigma_{imp}(K, T) , Dupire’s formula allows us to compute the local volatility function  $\sigma_{loc}(S, t)$ :


$$\sigma_{loc}^2(K, T) = \frac{\frac{\partial C}{\partial T} + rK \frac{\partial C}{\partial K}}{\frac{1}{2} K^2 \frac{\partial^2 C}{\partial K^2}}$$


where:
- $C(K, T$)  is the market price of a European call with strike  K  and maturity  T .
- $\frac{\partial C}{\partial T}$  is the time derivative of the call price.
- $\frac{\partial C}{\partial K}$  and  $\frac{\partial^2 C}{\partial K^2}$  are the first and second derivatives with respect to strike.

🔹 Interpretation:
- This formula tells us how the implied volatility surface evolves locally at each point.
- The result is a non-parametric, arbitrage-free volatility model.

### Generate synthetic data

In [2]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import interp2d

# Generate synthetic market option prices (Replace with real data)
strikes = np.linspace(80, 120, 10)  # Strike prices
maturities = np.linspace(0.1, 2, 10)  # Maturities in years

S=100
# Fake European call prices for different strikes & maturities
market_call_prices = np.array([[max(S - K, 0) * np.exp(-0.02 * T)  # Black-Scholes inspired
                                for K in strikes] for T in maturities])

# Fake implied volatility surface (Replace with real implied vol data)
market_implied_vols = np.random.rand(10, 10) * 0.2 + 0.1

### Compute Derivatives Using Finite Differences

In [3]:
def compute_derivatives(prices, strikes, maturities):
    """ Computes first and second derivatives of option prices. """
    dC_dT = np.gradient(prices, maturities, axis=0)  # ∂C/∂T
    dC_dK = np.gradient(prices, strikes, axis=1)  # ∂C/∂K
    d2C_dK2 = np.gradient(dC_dK, strikes, axis=1)  # ∂²C/∂K²

    return dC_dT, dC_dK, d2C_dK2

# Compute derivatives
dC_dT, dC_dK, d2C_dK2 = compute_derivatives(market_call_prices, strikes, maturities)

### Apply Dupire’s Formula

In [4]:
def compute_local_volatility(dC_dT, dC_dK, d2C_dK2, strikes, maturities, r=0.02):
    """ Computes local volatility using Dupire's formula. """
    local_vol = np.zeros_like(dC_dT)

    for i in range(len(maturities)):
        for j in range(len(strikes)):
            numerator = dC_dT[i, j] + r * strikes[j] * dC_dK[i, j]
            denominator = 0.5 * (strikes[j] ** 2) * d2C_dK2[i, j]
            if denominator > 0:
                local_vol[i, j] = np.sqrt(numerator / denominator)
            else:
                local_vol[i, j] = np.nan  # Avoid division errors
    
    return local_vol


In [5]:
# Compute Local Volatility Surface
local_vol_surface = compute_local_volatility(dC_dT, dC_dK, d2C_dK2, strikes, maturities)

/var/folders/bk/vr9m4cy108560n45r2h1hlh80000gn/T/ipykernel_22739/1711630112.py:10: RuntimeWarning: invalid value encountered in sqrt
  local_vol[i, j] = np.sqrt(numerator / denominator)


In [6]:
local_vol_surface

array([[nan, nan, nan, nan, nan, nan,  0., nan, nan, nan],
       [nan, nan, nan, nan, nan, nan,  0., nan, nan, nan],
       [nan, nan, nan, nan, nan, nan,  0., nan, nan, nan],
       [nan, nan, nan, nan, nan, nan,  0., nan, nan, nan],
       [nan, nan, nan, nan, nan, nan,  0., nan, nan, nan],
       [nan, nan, nan, nan, nan, nan,  0., nan, nan, nan],
       [nan, nan, nan, nan, nan, nan,  0., nan, nan, nan],
       [nan, nan, nan, nan, nan, nan,  0., nan, nan, nan],
       [nan, nan, nan, nan, nan, nan,  0., nan, nan, nan],
       [nan, nan, nan, nan, nan, nan,  0., nan, nan, nan]])